# IOAI — 2025 Contest Hogspell Challenge (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
# (diffusers/transformers 스택은 노트북 첫 셀이 /tmp/hogenv 에 격리 설치한다)
if not os.path.exists('data/prompts.json'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-hogspell-challenge', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
# 스티어링 코드(controller.py, steering_utils.py) — 레퍼런스(기본 브랜치=master)에서 받고 not_steer 버그 보정
for f in ['controller.py', 'steering_utils.py']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/lenjjiv/steering-hogs/master/{f}', f)
_s = open('steering_utils.py').read().replace('run_model(model_name, pipe, prompt, seed, num_denoising_steps, device)', 'run_model(pipe, prompt, seed, num_denoising_steps, device)')
open('steering_utils.py','w').write(_s)
print('데이터 준비:', 'data/prompts.json' if os.path.exists('data/prompts.json') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Hogspell · 모범답안 (Cross-Attention 스티어링)

**방법(CASteer)**: "pig" 개념이 있는/없는 프롬프트 쌍의 UNet self·cross-attention 활성 차이를 **스티어링 벡터**로
만들고(단계×블록별), 생성 중 각 블록의 hidden state 에 `α·v` 를 더해(노름 보존) **돼지 개념을 주입**합니다.
프롬프트·샘플링은 그대로 두고 **모델 내부 활성만** 조작합니다.

- `compute_steering_vectors(concept_pos="pig", concept_neg="")` 로 벡터 생성
- 생성 시 `steer_back=False, alpha=20` (**더하기 모드**) — α 가 클수록 강하게 돼지화(α=10→약함, α=20→강함)

스모크 검증: 기사 프롬프트의 돼지 확률(CLIP) 0.00 → α=20 에서 **0.99**. 캐글 점수: 베이스라인 ≈0 → 모범답안 ↑(리더보드 1위 ≈0.99).

### ⚙️ 먼저 실행 — 디퓨전 스택 격리 설치
이 cross-attention 스티어링은 **diffusers 0.31 + transformers 4.51.x** 에서 검증됐습니다(컨테이너 기본 transformers 5.x 는 diffusers 0.31 과 충돌). 검증 스택을 격리 디렉토리(`/tmp/hogenv`)에 설치하고 `sys.path` 최우선으로 로드합니다(시스템 패키지 영향 없음).

In [ ]:
import subprocess, sys
if '/tmp/hogenv' not in sys.path:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-script-location',
                    '--target', '/tmp/hogenv', '--no-deps',
                    'diffusers==0.31.0', 'transformers==4.51.3', 'tokenizers==0.21.1',
                    'huggingface_hub==0.30.2', 'accelerate==0.34.2'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    sys.path.insert(0, '/tmp/hogenv')
import diffusers, transformers; print('diffusers', diffusers.__version__, '| transformers', transformers.__version__)

In [ ]:
import sys, os
for p in ['data', '.']:
    if p not in sys.path: sys.path.insert(0, p)
import torch, json, base64, io, pandas as pd, time
from PIL import Image
from diffusers import StableDiffusionPipeline
from steering_utils import compute_steering_vectors, generate_image
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
# 모델: DGX 는 동봉 data/sd15(다운로드 불필요), Colab 등은 HF 에서 SD v1.5 다운로드
MODEL = 'data/sd15' if os.path.isdir('data/sd15') else 'sd-legacy/stable-diffusion-v1-5'
pipe = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, variant='fp16',
            safety_checker=None, requires_safety_checker=False).to(dev)
pipe.set_progress_bar_config(disable=True)
prompts = json.load(open('data/prompts.json' if os.path.exists('data/prompts.json') else 'prompts.json'))
def to_b64(img):
    buf = io.BytesIO(); img.save(buf, format='PNG'); return base64.b64encode(buf.getvalue()).decode()
print('prompts', len(prompts), '| model', MODEL)

In [ ]:
# 스티어링 벡터 생성 (pig vs 빈 프롬프트)
vecs = compute_steering_vectors(pipe, concept_pos='pig', concept_neg='', num_denoising_steps=25, max_prompts=12, n_times=1)
print('steering vectors:', len(vecs), 'steps')


In [ ]:
# 돼지 개념을 주입해 생성 (addition 모드, alpha=20)
t = time.time(); ids, b64s = [], []
for k, p in prompts.items():
    img = generate_image(pipe, p, steering_vectors=vecs, seed=42, alpha=20, steer_back=False, num_denoising_steps=25)
    ids.append(k); b64s.append(to_b64(img))
pd.DataFrame({'id': ids, '0': b64s}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(ids), round(time.time()-t,1),'s')


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)